<a href="https://colab.research.google.com/github/SarahSAH02/Dat255_prosjekt/blob/main/resnet_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision pandas pillow

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/archive (4).zip"
extract_path = "/content/chexpert"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped!")

Unzipped!


In [27]:
!ls /content/chexpert

train  train.csv  valid  valid.csv


In [28]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import pandas as pd
import os
from PIL import Image
from torchvision import transforms, models

In [32]:
ROOT = "/content/chexpert"

In [33]:
class CheXpertData(torch.utils.data.Dataset):
    def __init__(self, root_dir, csv_file, transform=None):
        self.root_dir = root_dir
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.root_dir, row["Path"].replace("CheXpert-v1.0-small/", ""))
        image = Image.open(img_path).convert("RGB")

        labels = row.iloc[5:].astype(float).fillna(0).replace(-1, 0).values
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels

In [34]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [44]:
train_dataset = CheXpertData(ROOT, f"{ROOT}/train.csv", train_transform)
valid_dataset = CheXpertData(ROOT, f"{ROOT}/valid.csv", valid_transform)

train_dataset = Subset(train_dataset, range(min(20000, len(train_dataset))))
valid_dataset = Subset(valid_dataset, range(min(3000, len(valid_dataset))))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32)

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [47]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 14)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [48]:
df = pd.read_csv(f"{ROOT}/train.csv")

labels = df.iloc[:, 5:].astype(float).fillna(0).replace(-1, 0)

pos_counts = labels.sum()
neg_counts = len(labels) - pos_counts

pos_weight = (neg_counts / (pos_counts + 1e-5)).clip(0, 20)
pos_weight = torch.tensor(pos_weight.values, dtype=torch.float32).to(device)

In [49]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [50]:
for epoch in range(2):
    model.train()

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

        if i % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {i} | Loss {loss.item():.4f}")

Epoch 1 | Batch 0 | Loss 1.2852
Epoch 1 | Batch 50 | Loss 1.2062
Epoch 1 | Batch 100 | Loss 0.9523
Epoch 1 | Batch 150 | Loss 0.9609
Epoch 1 | Batch 200 | Loss 1.0044
Epoch 1 | Batch 250 | Loss 0.9901
Epoch 1 | Batch 300 | Loss 0.9005
Epoch 1 | Batch 350 | Loss 1.0682
Epoch 1 | Batch 400 | Loss 1.0131
Epoch 1 | Batch 450 | Loss 1.0055
Epoch 1 | Batch 500 | Loss 0.9014
Epoch 1 | Batch 550 | Loss 0.7787
Epoch 1 | Batch 600 | Loss 0.9547
Epoch 2 | Batch 0 | Loss 1.0404
Epoch 2 | Batch 50 | Loss 0.8805
Epoch 2 | Batch 100 | Loss 1.0528
Epoch 2 | Batch 150 | Loss 0.7706
Epoch 2 | Batch 200 | Loss 0.9318
Epoch 2 | Batch 250 | Loss 1.0395
Epoch 2 | Batch 300 | Loss 0.9818
Epoch 2 | Batch 350 | Loss 0.9526
Epoch 2 | Batch 400 | Loss 0.8746
Epoch 2 | Batch 450 | Loss 0.9442
Epoch 2 | Batch 500 | Loss 1.0575
Epoch 2 | Batch 550 | Loss 0.8291
Epoch 2 | Batch 600 | Loss 0.8603


In [51]:
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pth")
print("Saved!")

Saved!


In [52]:
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

model.eval()

all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.sigmoid(outputs)

        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

In [53]:
threshold = 0.3
pred_binary = (all_preds > threshold).astype(int)

f1 = f1_score(all_labels, pred_binary, average="macro")
print("Macro F1:", f1)

Macro F1: 0.3966846814260589


In [55]:
import numpy as np
from sklearn.metrics import roc_auc_score

auc_list = []

for i in range(all_labels.shape[1]):
    try:
        auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
        if not np.isnan(auc):
            auc_list.append(auc)
    except:
        continue

if len(auc_list) > 0:
    print("Mean AUC:", np.mean(auc_list))
else:
    print("No valid AUC scores")

Mean AUC: 0.7307364372785558


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [56]:
import json

results = {
    "model": "ResNet18",
    "batch_size": 32,
    "epochs": 2,
    "f1_score": float(0.3966846814260589),
    "auc": float(0.7307364372785558)
}

with open("/content/drive/MyDrive/fast_results.json", "w") as f:
    json.dump(results, f)

print("Results saved!")

Results saved!


In [57]:
torch.save(model.state_dict(), "/content/drive/MyDrive/fast_model.pth")
print("Model saved!")

Model saved!


In [58]:
import numpy as np

np.save("/content/drive/MyDrive/preds.npy", all_preds)
np.save("/content/drive/MyDrive/labels.npy", all_labels)

print("Predictions saved!")

Predictions saved!


In [ ]:

train_dataset = CheXpertData(ROOT, f"{ROOT}/train.csv", train_transform)
valid_dataset = CheXpertData(ROOT, f"{ROOT}/valid.csv", valid_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 14)
model = model.to(device)

for param in model.parameters():
    param.requires_grad = True

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-5
)

best_val_loss = float("inf")

for epoch in range(3):


    model.train()
    train_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if i % 200 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss {loss.item():.4f}")

    train_loss /= len(train_loader)



    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

    val_loss /= len(valid_loader)



    print("\n" + "="*40)
    print(f"Epoch {epoch+1}/3")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print("="*40)


    # 💾 Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_model.pth")
        print("✅ Best model saved!")

print("training complete!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 | Batch 0/13964 | Loss 1.3241
Epoch 1 | Batch 200/13964 | Loss 0.9994
Epoch 1 | Batch 400/13964 | Loss 0.9252
Epoch 1 | Batch 600/13964 | Loss 1.0061
Epoch 1 | Batch 800/13964 | Loss 0.9883
Epoch 1 | Batch 1000/13964 | Loss 0.8797
Epoch 1 | Batch 1200/13964 | Loss 0.8718
Epoch 1 | Batch 1400/13964 | Loss 1.0080
Epoch 1 | Batch 1600/13964 | Loss 1.0965
Epoch 1 | Batch 1800/13964 | Loss 1.0108
Epoch 1 | Batch 2000/13964 | Loss 1.0676
Epoch 1 | Batch 2200/13964 | Loss 0.7969
Epoch 1 | Batch 2400/13964 | Loss 1.0838
Epoch 1 | Batch 2600/13964 | Loss 0.7692
Epoch 1 | Batch 2800/13964 | Loss 0.7188
Epoch 1 | Batch 3000/13964 | Loss 0.7280
Epoch 1 | Batch 3200/13964 | Loss 0.9938
Epoch 1 | Batch 3400/13964 | Loss 0.9042
Epoch 1 | Batch 3600/13964 | Loss 0.9434
Epoch 1 | Batch 3800/13964 | Loss 0.6361
Epoch 1 | Batch 4000/13964 | Loss 0.6617
Epoch 1 | Batch 4200/13964 | Loss 0.7842
Epoch 1 | Batch 4400/13964 | Loss 0.8110
Epoch 1 | Batch 4600/13964 | Loss 1.1508
Epoch 1 | Batch 4800/13